In [ ]:
from visualization import *

In [ ]:
from matrices import *

nx1, ny1 = 0, -1
nx2 , ny2 = 1, 0
nx3, ny3 = 0, 1
nx4, ny4 = -1, 0

x1, y1 = 0, -9

x2, y2 = 5, 0

x3, y3 = 0, 5

x4, y4 = -9, 0

points = np.array([[x1, y1], [x2, y2], [x3, y3], [x4, y4]], dtype=np.float64)
plt.title('Parametric Spline')
matplotlib_visualize_splines(points, 'Parametric Spline from scipy', 'Control Points: parametric spline from scipy', True)
plt.show()

In [ ]:
from perception_data import Centreline

centreline = Centreline(4, points, 0.5*np.ones(4), 0.2)

plt.title('Race track centreline')
implemented_visualize_splines(centreline.p, 'Parametric Spline: from implementation', 'Control Points', True)
plt.show()

In [ ]:
import cvxpy as cp

In [ ]:
from solve_qp import solve_for_alpha

alpha = solve_for_alpha(centreline)

In [ ]:
visualize_solution(centreline, points_raceline(centreline, alpha))

In [ ]:
from export_solution import export_solution

export_solution(points_raceline(centreline, alpha), 'waypoints3.json', format='json')

In [ ]:
from csv_reader import centreline_from_csv

centreline = centreline_from_csv(1.0, '../../maps/Spielberg_centerline.csv')
# centreline = centreline_from_csv(1.0, '../../maps/Budapest_centerline.csv')

# Run the last 4 cells again to visualize the raceline and export the waypoints to a json file

## `practicemap` trial for F1Tenth CDC 2024:

In [ ]:
#TODO: read parameters from practicemap.yaml into variables and use them
import pandas as pd
import numpy as np

'''
Resolved: in meters
(unspecified): in grid units
'''

df = pd.read_csv("../../maps/practicemap/practicemap.csv")
centerline_points_resolved = df.values
centerline_points = np.array((df.values + [-3.63, -8.52])/0.05, dtype=np.int32)

In [ ]:
VEHICLE_WIDTH = 0.27

In [ ]:
import cv2 as cv
practicemap = cv.imread("../../maps/practicemap/practicemap.pgm", cv.IMREAD_GRAYSCALE)

# for point in centerline_points:
#     practicemap[point[1], point[0]] = 1
# cv.imshow("practicemap", practicemap)
# cv.waitKey(0)
# cv.destroyAllWindows()

In [ ]:
from perception_data import Centreline
from matrices import *

def bresenham_measure_width(map: np.ndarray, point1: np.ndarray, point2: np.ndarray, occ_thresh: int, practicemap_scratch: np.ndarray):
    #TODO: remove practicemap_scratch after debugging
    x1, y1 = point1
    x2, y2 = point2
    dx = abs(x2 - x1)
    dy = abs(y2 - y1)
    if x1 < x2:
        sx = 1
    else:
        sx = -1
    if y1 < y2:
        sy = 1
    else:
        sy = -1
    err = dx - dy
    while True:
        if 0 <= y1 < map.shape[0] and 0 <= x1 < map.shape[1]:
            if map[y1, x1] < 255 - occ_thresh:
                return np.linalg.norm(np.array([x1, y1]) - point1)
            practicemap_scratch[y1, x1] = 1
        if x1 == x2 and y1 == y2:
            break
        e2 = 2*err
        if e2 > -dy:
            err = err - dy
            x1 = x1 + sx
        if e2 < dx:
            err = err + dx
            y1 = y1 + sy
    return np.linalg.norm(np.array([x1, y1]) - point1)

def track_width(points: np.ndarray, map: np.ndarray, occ_thresh: int = 50):
    # points: unresolved
    # returned track_width: resolved
    map_scratch = map.copy() #debug
    Ainv = matAInv(points.shape[0])
    centreline = Centreline(points.shape[0], points, None, VEHICLE_WIDTH)
    q_x = q_comp(centreline, 0)
    q_y = q_comp(centreline, 1)
    x_d = first_derivatives(centreline, Ainv, q_x)
    y_d = first_derivatives(centreline, Ainv, q_y)
    centreline.calc_n(x_d, y_d)
    
    track_widths = []
    for i in range(centreline.N):
        x, y = points[i]
        n_x, n_y = centreline.n[i]

        # TODO: check whether left_endpoint and right_endpoint are actually on the left and right sides
        _, t_left, t_right, _ = sorted([-x/n_x, (map.shape[1] - x)/n_x, -y/n_y, (map.shape[0] - y)/n_y])
        left_endpoint = np.array([x + t_left*n_x, y + t_left*n_y], dtype=np.int32)
        right_endpoint = np.array([x + t_right*n_x, y + t_right*n_y], dtype=np.int32)

        tr_w_left = bresenham_measure_width(map, points[i], left_endpoint, occ_thresh, map_scratch)
        tr_w_right = bresenham_measure_width(map, points[i], right_endpoint, occ_thresh, map_scratch)
        track_width = (tr_w_left + tr_w_right) / 2
        track_widths.append(track_width)
    track_widths_resolved = np.array(track_widths, dtype=np.float32)*0.05
    print(track_widths_resolved) #debug
    print(centreline.n) #debug
    cv.imshow("map_scratch", map_scratch)
    cv.waitKey(0)
    cv.destroyAllWindows()
    return track_widths_resolved

centreline = Centreline(centerline_points.shape[0], centerline_points_resolved, track_width(centerline_points, practicemap), VEHICLE_WIDTH)


In [ ]:
alpha = solve_for_alpha(centreline)

In [ ]:
points_boundary_max = centreline.p + centreline.n * np.repeat(centreline.half_w_tr, 2, axis=0).reshape((-1, 2)) 
points_boundary_min = centreline.p - centreline.n * np.repeat(centreline.half_w_tr, 2, axis=0).reshape((-1, 2)) 
# implemented_visualize_splines(centreline.p, 'Centreline from implementation', 'Control Points: Centreline from implementation', dashed=True)
# implemented_visualize_splines(points_boundary_max, 'Outer boundary from implementation', 'Control Points: Outer boundary from implementation')
# implemented_visualize_splines(points_boundary_min, 'Inner boundary from implementation', 'Control Points: Inner boundary from implementation')

matplotlib_visualize_splines(centreline.p, 'Centreline from scipy', 'Control Points: Centreline from scipy', True)
# matplotlib_visualize_splines(points_boundary_max, 'Outer boundary from scipy', 'Control Points: Outer boundary from scipy', True)
# matplotlib_visualize_splines(points_boundary_min, 'Inner boundary from scipy', 'Control Points: Inner boundary from scipy', True)
plt.show()

In [ ]:
visualize_solution(centreline, points_raceline(centreline, alpha))
export_solution(points_raceline(centreline, alpha), 'waypoints3.csv', format='csv')

In [ ]:
raceline_resolved = points_raceline(centreline, alpha)
raceline = np.array((raceline_resolved + [-3.63, -8.52])/0.05, dtype=np.int32)
practicemap_scratch = practicemap.copy()
for point in raceline:
    practicemap_scratch[point[1], point[0]] = 1
cv.imshow("practicemap_scratch", practicemap_scratch)
cv.waitKey(0)
cv.destroyAllWindows()